# 01 - Audit du jeu de données

Inspection de l'échantillon RSNA prétraité : répartition des classes, statistiques de features et aperçu d'une image. Aucune donnée patient n'est affichée.

> Prototype pédagogique. Non destiné au diagnostic.

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
while not (root / 'pyproject.toml').exists() and root != root.parent:
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root))
print('Racine du projet:', root)

In [ ]:
import collections, csv
with open('data/raw/rsna_sample_labels.csv', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
counts = collections.Counter(r['label'] for r in rows)
print('Cas:', len(rows))
for label, n in sorted(counts.items()):
    print(f'  {label:18} {n}')

In [ ]:
import statistics as st
from collections import defaultdict
from src.webapp.repository import PredictionRepository

repo = PredictionRepository('data/predictions/baseline_v1')
by_class = defaultdict(list)
for case in repo.list_cases():
    by_class[case.label].append(case.features)

for feature in ('std_intensity', 'central_bright_ratio', 'horizontal_asymmetry', 'local_texture_max'):
    line = f'{feature:22}'
    for cls in ('normal', 'suspected_opacity', 'uncertain'):
        vals = [c[feature] for c in by_class[cls]]
        line += f' | {cls}: {st.mean(vals):.2f} ({min(vals):.2f}..{max(vals):.2f})'
    print(line)

`local_texture_max` (contraste local maximal) sépare le mieux `normal` (texture vasculaire élevée) de `suspected_opacity` (consolidation homogène). C'est le signal exploité par la variante améliorée (notebook 03).

In [ ]:
from PIL import Image
cases = repo.list_cases()
image_path = repo.resolve_image_path(cases[0])
print('Aperçu:', cases[0].case_id, '->', image_path)
Image.open(image_path)